In [1]:
import numpy as np
import pandas as pd 
import time
from tqdm.auto import tqdm

In [47]:
# Load up the stream history from 2026
full_streaming_history = pd.read_json('./spotify_data/Streaming_History_Audio_2026.JSON')

# Define the columns to use and their respective re-names
columns = ['spotify_track_uri',
            'master_metadata_track_name',
            'master_metadata_album_album_name',
            'master_metadata_album_artist_name',
            'ms_played',
            'ts',
            'skipped']

re_names = ['track_id',
            'track_name',
            'album_name',
            'artist_name',
            'stream_time_ms',
            'time_stamp',
            'skipped']

# Get the tracks stream information into a dataframe
track_streams_df = full_streaming_history[columns].rename(columns = dict(zip(columns, re_names))).dropna()
track_streams_df.head()



,track_id,track_name,album_name,artist_name,stream_time_ms,time_stamp,skipped
0,spotify:track:5icLTeOSXTvkLIi08P2k4g,The Day That Never Comes - Live,S&M2,Metallica,192580,2026-01-08T14:15:33Z,True
1,spotify:track:0msDFRusep31oGvYA4WOal,Fuel - Live,Metallica Through The Never,Metallica,237427,2026-01-08T14:19:30Z,False
2,spotify:track:4b224oNVsda1doe1CQAwou,Duality - Live,Day Of The Gusano,Slipknot,194273,2026-01-08T14:24:17Z,False
3,spotify:track:7tcaQvBbSpNN0P2UF6WXxB,Children of the Damned - 2015 Remaster,The Number of the Beast,Iron Maiden,3070,2026-01-08T17:47:57Z,True
4,spotify:track:6olS0TmHmsGr0hXtcBsiVM,A Tout Le Monde - Remastered 2004,Youthanasia,Megadeth,1886,2026-01-08T17:47:59Z,True


In [48]:
# Assign an unique Artist ID to each artist
unique_artists = track_streams_df.sort_values(by='artist_name')['artist_name'].unique()
unique_artist_ids = range(unique_artists.size)

artists_and_ids = {'id': unique_artist_ids,
                   'name': unique_artists}

# Get the artists and their ids into a dataframe
artists_df = pd.DataFrame(artists_and_ids)
artists_df.head()

,id,name
0,0,*NSYNC
1,1,100 gecs
2,2,31 Minutos
3,3,311
4,4,3BallMTY


In [49]:
tracks_df = track_streams_df[['track_id','track_name','album_name','artist_name']].drop_duplicates(subset='track_id').reset_index(drop=True)
tracks_df.columns = ['id','name','album','artist']
tracks_df.head()

,id,name,album,artist
0,spotify:track:5icLTeOSXTvkLIi08P2k4g,The Day That Never Comes - Live,S&M2,Metallica
1,spotify:track:0msDFRusep31oGvYA4WOal,Fuel - Live,Metallica Through The Never,Metallica
2,spotify:track:4b224oNVsda1doe1CQAwou,Duality - Live,Day Of The Gusano,Slipknot
3,spotify:track:7tcaQvBbSpNN0P2UF6WXxB,Children of the Damned - 2015 Remaster,The Number of the Beast,Iron Maiden
4,spotify:track:6olS0TmHmsGr0hXtcBsiVM,A Tout Le Monde - Remastered 2004,Youthanasia,Megadeth


In [54]:
# Add the artist id to the streams dataframe
track_streams_df["artist_id"] = [artists_df[artists_df['name'] == artist]['id'].values[0] for artist in track_streams_df['artist_name'].values]

# Clean up the streams table 
streams_table_columns = ['track_id','artist_id','stream_time_ms','time_stamp','skipped']
cleaned_streams_df = track_streams_df[streams_table_columns].dropna().reset_index(drop=True)
cleaned_streams_df.head()

,track_id,artist_id,stream_time_ms,time_stamp,skipped
0,spotify:track:5icLTeOSXTvkLIi08P2k4g,338,192580,2026-01-08T14:15:33Z,True
1,spotify:track:0msDFRusep31oGvYA4WOal,338,237427,2026-01-08T14:19:30Z,False
2,spotify:track:4b224oNVsda1doe1CQAwou,453,194273,2026-01-08T14:24:17Z,False
3,spotify:track:7tcaQvBbSpNN0P2UF6WXxB,233,3070,2026-01-08T17:47:57Z,True
4,spotify:track:6olS0TmHmsGr0hXtcBsiVM,337,1886,2026-01-08T17:47:59Z,True


In [55]:
# Save the dataframes to CSV files
artists_df.to_csv('./artists.csv', index=False)
tracks_df.to_csv('./tracks.csv', index=False)
cleaned_streams_df.to_csv('./streams.csv', index=False)